# 01 - Dataset Discovery

Profile source folders and confirm files/labels before preprocessing.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'rareguard').exists()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as exc:
            print('Google Drive mount skipped/failed:', exc)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. '
        'If you are using Colab connector, Colab cannot read /Volumes/LocalDrive1 directly. '
        'Copy/sync the IMPLEMENTATION folder to Google Drive at '
        '/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION, '
        'or run this notebook with a local VS Code Python kernel on your Mac.'
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python path ready')
for name in ['logs', 'results', 'processed', 'features', 'models', 'figures', 'reports']:
    (PROJECT_ROOT / 'artifacts' / name).mkdir(parents=True, exist_ok=True)
print('Artifact folders ready:', PROJECT_ROOT / 'artifacts')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Python path ready
Artifact folders ready: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts


In [2]:
!python -m rareguard.experiments.dataset_discovery --config configs/default.yaml --datasets configs/datasets.yaml


2026-09-25 03:28:28,991 | INFO | rareguard | Wrote dataset profiles to artifacts/results/dataset_profiles.csv
2026-09-25 03:28:28,996 | INFO | rareguard | CSE-CIC-IDS2018 files=948 bytes=539.453 GiB labels=['Label']
2026-09-25 03:28:28,996 | INFO | rareguard | CIC-BCCC-NRC-TabularIoTAttack-2024 files=17 bytes=1.845 GiB labels=['Label']
2026-09-25 03:28:28,996 | INFO | rareguard | CIC-IIoT-2025-DataSense files=43 bytes=3.588 GiB labels=['label1', 'label2', 'label3', 'label4', 'label_full']
2026-09-25 03:28:28,996 | INFO | rareguard | Edge-IIoTset files=5 bytes=1.648 GiB labels=['Attack_label', 'Attack_type']


In [3]:
import pandas as pd
path = 'artifacts/results/dataset_profiles.csv'
df = pd.read_csv(path)
display(df)


,key,folder,path,exists,manifest_status,file_count,csv_count,parquet_count,archive_count,bytes_total,label_candidates,detected_label_columns
0,cse_cic_ids2018,CSE-CIC-IDS2018,/content/drive/MyDrive/projects/Dataset/Securi...,True,downloaded,948,10,0,20,579233204801,"['Label', 'label', 'Attack', 'attack', 'Class'...",['Label']
1,cic_bccc_nrc_tabulariotattack_2024,CIC-BCCC-NRC-TabularIoTAttack-2024,/content/drive/MyDrive/projects/Dataset/Securi...,True,downloaded,17,15,0,1,1980622662,"['Label', 'label', 'Attack', 'attack', 'Class'...",['Label']
2,cic_iiot_2025_datasense,CIC-IIoT-2025-DataSense,/content/drive/MyDrive/projects/Dataset/Securi...,True,downloaded,43,20,0,1,3852147919,"['label_full', 'label1', 'label2', 'label3', '...","['label1', 'label2', 'label3', 'label4', 'labe..."
3,edge_iiotset,Edge-IIoTset,/content/drive/MyDrive/projects/Dataset/Securi...,True,downloaded,5,3,0,1,1769143768,"['Attack_label', 'Attack_type', 'attack_type',...","['Attack_label', 'Attack_type']"


In [4]:
# Optional: sync artifacts to the configured Google Drive folder.
# Keep --dry-run first if you want to preview.
!python scripts/sync_to_drive.py --dry-run
# !python scripts/sync_to_drive.py


rclone is not installed in this runtime.
Artifacts are already being written inside Google Drive because the project root is under /content/drive.
Current artifact folder: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts
Skipping external rclone mirror. Continue with the next notebook.
